In [1]:
import math
import os
from copy import deepcopy

def count_points(space):
    """
    Calcula el número total de puntos en el espacio.
    Cada parámetro contribuye con:
        n = floor((max - min)/step) + 1
    """
    total = 1
    for param, limits in space.items():
        pmin = limits["min"]
        pmax = limits["max"]
        step = limits["step"]
        n = int(math.floor((pmax - pmin) / step)) + 1
        total *= n
    return total

def split_parameter(space, param, splits):
    """
    Dado un espacio y un parámetro, divide el rango de dicho parámetro
    en 'splits' subintervalos de tamaño aproximadamente igual.
    Devuelve una lista de nuevos espacios, cada uno con el subintervalo correspondiente.
    """
    pmin = space[param]["min"]
    pmax = space[param]["max"]
    step = space[param]["step"]
    
    subspaces = []
    for i in range(splits):
        # División lineal del intervalo
        sub_min = pmin + i * (pmax - pmin) / splits
        sub_max = pmin + (i + 1) * (pmax - pmin) / splits
        # Se hace una copia profunda del espacio y se actualiza el parámetro a subdividir.
        new_space = deepcopy(space)
        new_space[param] = {"min": sub_min, "max": sub_max, "step": step}
        subspaces.append(new_space)
    return subspaces

def partition_space(space, allowed_points, param_order):
    """
    Función recursiva que subdivide el espacio de parámetros
    hasta que el número de puntos de la grilla sea menor o igual a allowed_points.
    
    param_order: lista de parámetros en el orden preferente para dividir.
                 Puede ser la jerarquía predeterminada o determinada a partir de pesos.
    """
    current_points = count_points(space)
    if current_points <= allowed_points:
        return [space]
    
    # Recorremos los parámetros según el orden definido
    for param in param_order:
        # Si el parámetro existe y su rango tiene más de un punto:
        if param in space:
            pmin = space[param]["min"]
            pmax = space[param]["max"]
            step = space[param]["step"]
            n_param = int(math.floor((pmax - pmin) / step)) + 1
            if n_param > 1:
                # Calcula el total de puntos de los otros parámetros.
                others = 1
                for other_param, limits in space.items():
                    if other_param == param:
                        continue
                    others *= (int(math.floor((limits["max"] - limits["min"]) / limits["step"])) + 1)
                # Para que cada subespacio cumpla: (n_param / splits) * others <= allowed_points,
                # se requiere que splits >= (n_param * others) / allowed_points.
                needed_splits = math.ceil((n_param * others) / allowed_points)
                # Nos aseguramos de tener al menos 2 subdivisiones y no más que n_param.
                splits = max(2, min(n_param, needed_splits))
                # Dividir el espacio para este parámetro:
                subspaces = split_parameter(space, param, splits)
                result = []
                for subspace in subspaces:
                    result.extend(partition_space(subspace, allowed_points, param_order))
                return result
    # Si ningún parámetro se puede dividir más, se devuelve el espacio actual.
    return [space]

def write_config_file(space, filename, param_order):
    """
    Escribe un archivo .conf con el formato:
        param_min value
        param_max value
        step_param value
    siguiendo el orden en param_order.
    """
    lines = []
    for param in param_order:
        if param in space:
            limits = space[param]
            lines.append(f"{param}_min {limits['min']}")
            lines.append(f"{param}_max {limits['max']}")
            lines.append(f"step_{param} {limits['step']}")
            lines.append("")  # línea en blanco para separar bloques
    with open(filename, "w") as f:
        f.write("\n".join(lines))
    print(f"Archivo generado: {filename}  -> {count_points(space)} puntos.")


## Espacio parametrico y configs

In [19]:
import numpy as np
# Espacio de parámetros (ejemplo basado en el archivo .conf mostrado)
param_space = {
    "lambda6":    {"min": -3,    "max": 3,      "step": 0.33},
    "lambda7":    {"min": -3,    "max": 3,      "step": 0.33},
    "m12_squared":{"min": -3,   "max": 3,     "step": 0.33},
    "alpha":      {"min": -0.9,"max": 0.9,   "step": 0.33},
    "beta":       {"min": 0.9, "max": np.pi/2, "step": 0.028510001},
    "mphi":       {"min": 125,   "max": 400,    "step": 28.314},
    "mA":         {"min": 320.0, "max": 500.0,  "step": 44.3}
}

# Tiempo de cómputo por punto (en minutos)
# Por ejemplo: 3000 puntos toman 20 minutos => 20/3000
time_per_point = 1 / 15000

print(f"Espacio total de puntos: { count_points(param_space) : .2e}")
print(f"Tiempo total estimado: { count_points(param_space) * time_per_point / 60 / 24 } dias")


# Orden de parámetros basado en la jerarquía (también se podría ajustar según pesos)
default_order = ["lambda6", "lambda7", "m12_squared", "alpha", "beta", "mphi", "mA"]



Espacio total de puntos:  4.94e+07
Tiempo total estimado: 2.2863333333333333 dias


In [4]:
# borrar las configs previas
!rm -r ../../../dihiggs/app/configs_ordered/

## Computo

In [24]:
# Tiempo objetivo para cada trabajo (en minutos)
target_time = 80  # por ejemplo, queremos que cada .conf genere un trabajo de 5 minutos

# Número máximo de puntos por archivo
allowed_points = target_time / time_per_point

# ========== logs ==========
print("Espacio total de puntos:", count_points(param_space))
print("Puntos máximos permitidos por archivo:", allowed_points)


# Si se desea incluir un diccionario de pesos, podría hacerse algo como:
# weights = {"lambda6": 0.8, "lambda7": 0.8, "m12_squared": 0.7, "alpha": 1.0, "beta": 1.0, "mphi": 0.5, "mA": 0.5}
# Y luego ordenar default_order en función de esos pesos (por ejemplo, priorizando los de menor peso para dividir primero).
# Aquí se deja como ejercicio ampliar el algoritmo para usar esos pesos.

# Particionar el espacio en subespacios que cumplan la condición de puntos
subspaces = partition_space(param_space, allowed_points, default_order)
print(f"Se generarán {len(subspaces)} archivos de configuración.")

Espacio total de puntos: 49384800
Puntos máximos permitidos por archivo: 1200000.0
Se generarán 57 archivos de configuración.


In [25]:
# Crear una carpeta para guardar los archivos (si no existe)
output_dir = "/mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404"
os.makedirs(output_dir, exist_ok=True)

In [26]:
# Escribir cada subespacio en un archivo .conf
for idx, space in enumerate(subspaces):
    filename = os.path.join(output_dir, f"config_{idx}.conf")
    write_config_file(space, filename, default_order)


Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_0.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_1.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_2.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_3.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_4.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_5.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/config_6.conf  -> 957600 puntos.
Archivo generado: /mnt/c/Users/fbien.DESKTOP-6FMEAR7/Desktop/mlclassic_kosmos/config_dbeta_0404/c